# Dog Vision Research Paper — Cloud Experiments on Kaggle

This notebook reproduces the core validation and calibration experiments for the dog breed detection research paper.

**What it runs**
1. `paper/scripts/exp1_validation_suite.py` — full Stanford Dogs validation set evaluation (20,580 images, 120 breeds).
2. `paper/scripts/exp3_calibration.py` — reliability diagrams and calibration metrics.
3. `paper/scripts/exp1_figures.py` — data-dependent figures (confusion matrix, agreement heatmap, per-breed accuracy).

**Prerequisites**
- Upload `models/` and `data/Images/` as Kaggle datasets **or** clone the GitHub repo.
- Enable a GPU/TPU accelerator for reasonable runtime (CPU works but is slow).

**Outputs**
- `paper/results/` — metrics, predictions, confusion matrices, CSVs.
- `paper/figures/` — publication-ready figures.
- Two zip files are created at the end for easy download.

## 1. Install dependencies

Kaggle's environment sometimes fails on the `uvicorn[standard]` extra (it relies on compiled packages that may not be available). We install everything else and fall back to plain `uvicorn` if needed.

In [ ]:
import os
import subprocess
import sys

# Prefer repo requirements.txt if present, otherwise hard-code the list
req_file = "requirements.txt"
if os.path.exists(req_file):
    with open(req_file) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("#")]
else:
    lines = [
        "fastapi==0.139.0",
        "uvicorn[standard]==0.49.0",
        "python-multipart==0.0.20",
        "tensorflow==2.21.0",
        "tensorflow-hub==0.16.1",
        "numpy==2.4.6",
        "pillow==11.3.0",
        "requests==2.32.4",
    ]

# Skip uvicorn[standard]; install remaining packages
deps = [line for line in lines if "uvicorn[standard]" not in line]
uvicorn_spec = next((line for line in lines if "uvicorn[standard]" in line), "uvicorn==0.49.0")
uvicorn_plain = uvicorn_spec.split("[" )[0] + "==" + uvicorn_spec.split("==")[-1]

print("Installing core dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + deps, check=True)

# Try plain uvicorn (the [standard] extra is only needed for local server performance)
print("Installing uvicorn (plain)...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", uvicorn_plain], check=False)

print("\nDependency installation complete.")

## 2. Locate or clone project assets

The notebook needs three things in the same directory:
- `models/` — the two `.keras` model files plus temperature JSONs.
- `data/Images/` — the Stanford Dogs image folders.
- `data/unique_breeds.json` — breed label list.

We first check the current directory, then search `/kaggle/input/` for datasets named after the project or common asset names, and finally fall back to cloning the GitHub repository.

In [ ]:
import glob
import shutil
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path.cwd()
ON_KAGGLE = os.path.exists("/kaggle/input")

def find_dir_under(start: Path, name: str, must_contain=None):
    """Breadth-first search for a directory named `name` under `start`."""
    for root, dirs, _ in os.walk(start):
        for d in dirs:
            if d == name:
                candidate = Path(root) / d
                if must_contain is None or any(candidate.glob(must_contain)):
                    return candidate
    return None

def ensure_assets():
    """Make sure models/ and data/Images/ exist in PROJECT_ROOT."""
    models_dir = PROJECT_ROOT / "models"
    images_dir = PROJECT_ROOT / "data" / "Images"
    labels_file = PROJECT_ROOT / "data" / "unique_breeds.json"

    # If repo is already complete, nothing to do
    if models_dir.exists() and images_dir.exists() and labels_file.exists():
        print("Assets already present in current directory.")
        return

    if ON_KAGGLE:
        input_root = Path("/kaggle/input")
        print(f"Searching Kaggle input datasets under {input_root}...")

        # Common dataset names users might use
        for ds_name in ["dog-vision-back", "dog-vision-assets", "dog-vision-models", "stanford-dogs"]:
            ds_path = input_root / ds_name
            if not ds_path.exists():
                continue
            print(f"  Found dataset: {ds_path}")
            if not models_dir.exists():
                cand = find_dir_under(ds_path, "models", "*.keras")
                if cand:
                    shutil.copytree(cand, models_dir, dirs_exist_ok=True)
                    print(f"    -> copied models to {models_dir}")
            if not images_dir.exists():
                cand = find_dir_under(ds_path, "Images")
                if cand:
                    (PROJECT_ROOT / "data").mkdir(parents=True, exist_ok=True)
                    shutil.copytree(cand, images_dir, dirs_exist_ok=True)
                    print(f"    -> copied images to {images_dir}")
            if not labels_file.exists():
                cand = find_dir_under(ds_path, "data", "unique_breeds.json")
                if cand and (cand / "unique_breeds.json").exists():
                    (PROJECT_ROOT / "data").mkdir(parents=True, exist_ok=True)
                    shutil.copy(cand / "unique_breeds.json", labels_file)
                    print(f"    -> copied labels to {labels_file}")
                elif list(ds_path.glob("unique_breeds.json")):
                    (PROJECT_ROOT / "data").mkdir(parents=True, exist_ok=True)
                    shutil.copy(ds_path / "unique_breeds.json", labels_file)
                    print(f"    -> copied labels to {labels_file}")

    # Final fallback: clone the repo (models/images still need to be supplied by the user)
    if not (PROJECT_ROOT / "paper" / "scripts" / "exp1_validation_suite.py").exists():
        repo_url = "https://github.com/MozzamShahid/dog-vision-back.git"
        print(f"Cloning repository from {repo_url}...")
        subprocess.run(["git", "clone", "--depth", "1", repo_url, "dog-vision-back"], check=True)
        # Move repo contents up one level
        cloned = PROJECT_ROOT / "dog-vision-back"
        for item in cloned.iterdir():
            dest = PROJECT_ROOT / item.name
            if item.is_dir():
                shutil.copytree(item, dest, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dest)
        shutil.rmtree(cloned)

    # Sanity checks
    if not models_dir.exists() or not list(models_dir.glob("*.keras")):
        raise FileNotFoundError(
            "models/ is missing or contains no .keras files. "
            "Please upload models/ as a Kaggle dataset and re-run."
        )
    if not images_dir.exists():
        raise FileNotFoundError(
            "data/Images/ is missing. Please upload the Stanford Dogs Images/ folder as a Kaggle dataset."
        )
    if not labels_file.exists():
        raise FileNotFoundError("data/unique_breeds.json is missing.")

    print(f"\nAssets ready:")
    print(f"  models: {models_dir} ({len(list(models_dir.glob('*.keras')))} .keras files)")
    print(f"  images: {images_dir}")
    print(f"  labels: {labels_file}")

ensure_assets()

## 3. Run the validation suite (exp1)

This evaluates the full Stanford Dogs dataset and writes results to `paper/results/`.

Expected runtime: ~10–30 min on GPU, much longer on CPU. To run a quick smoke test, add `--max-images 500` to the command below.

In [ ]:
subprocess.run([
    sys.executable, "paper/scripts/exp1_validation_suite.py",
    "--models-dir", "models",
    "--images-dir", "data/Images",
    "--labels-path", "data/unique_breeds.json",
    "--output-dir", "paper/results",
], cwd=PROJECT_ROOT, check=True)

## 4. Run calibration analysis (exp3)

Produces reliability diagrams, confidence histograms, and `calibration_metrics.csv`. Requires the `predictions.npz` created in the previous step.

In [ ]:
subprocess.run([
    sys.executable, "paper/scripts/exp3_calibration.py",
    "--predictions", "paper/results/predictions.npz",
    "--output-dir", "paper/results",
    "--figures-dir", "paper/figures",
], cwd=PROJECT_ROOT, check=True)

## 5. Generate data-dependent figures (exp1 figures)

Creates the top-20 confusion matrix, model agreement heatmap, and sorted per-breed accuracy bar chart.

In [ ]:
subprocess.run([
    sys.executable, "paper/scripts/exp1_figures.py",
    "--results-dir", "paper/results",
    "--figures-dir", "paper/figures",
    "--labels-path", "data/unique_breeds.json",
], cwd=PROJECT_ROOT, check=True)

## 6. Package results for download

Zip `paper/results/` and `paper/figures/` so they can be downloaded from the Kaggle output panel.

In [ ]:
import zipfile

def zip_directory(source: Path, archive: Path):
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in source.rglob("*"):
            if file.is_file():
                zf.write(file, file.relative_to(PROJECT_ROOT))
    print(f"Created {archive} ({archive.stat().st_size / 1e6:.1f} MB)")

zip_directory(PROJECT_ROOT / "paper/results", PROJECT_ROOT / "paper_results.zip")
zip_directory(PROJECT_ROOT / "paper/figures", PROJECT_ROOT / "paper_figures.zip")

print("\nAll experiments complete. Download paper_results.zip and paper_figures.zip from the Output panel.")

## Appendix: quick sanity check

List the produced files and show the main metrics.

In [ ]:
import json

print("Result files:")
for p in sorted((PROJECT_ROOT / "paper/results").rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(PROJECT_ROOT))

print("\nFigure files:")
for p in sorted((PROJECT_ROOT / "paper/figures").rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(PROJECT_ROOT))

metrics_path = PROJECT_ROOT / "paper/results/metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("\nMetrics summary:")
    print(json.dumps(metrics["ensemble_calibrated"], indent=2))